In [ ]:
import os
from os.path import join
from pathlib import Path

import kagglehub
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, \
    confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree

kagglehub.login()

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

In [ ]:
random_state=42
max_depth=3

In [ ]:
os.listdir(path)

In [ ]:
X_full = pd.read_csv(Path(join(path, 'train.csv')))
X_test = pd.read_csv(Path(join(path, 'test.csv')))

In [ ]:
len(X_full)

In [ ]:
X_full.head()

In [ ]:
def describe_columns(column_name: str, column_seq: pd.Series):
    print(f'Column: {column_name}')
    print(f'Type: {column_seq.dtype}')
    print(f"Has NANs: {column_seq.hasnans}")
    print(" ")

In [ ]:
describe_columns('Survived', X_full.Survived)
describe_columns('Pclass', X_full.Pclass)
describe_columns('Sex', X_full.Sex)
describe_columns('Age', X_full.Age)
describe_columns('SibSp', X_full.SibSp)
describe_columns('Parch', X_full.Parch)
describe_columns('Ticket', X_full.Ticket)
describe_columns('Fare', X_full.Fare)
describe_columns('Cabin', X_full.Cabin)
describe_columns('Embarked', X_full.Embarked)


In [ ]:
X_test.head()

In [ ]:
y = X_full.Survived
X = X_full.drop(['Survived', 'PassengerId', 'Name'], axis=1)
X_test = X_test.drop(['Name'], axis=1)

In [ ]:
# drop columns with missing values for now
cols_with_missing = [col for col in X.columns if X[col].isnull().any()]
cols_with_missing

In [ ]:
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8,
                                                      test_size=0.2,
                                                      random_state=0)

In [ ]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])
drop_X_test = X_test.select_dtypes(exclude=['object'])
print(drop_X_train.columns)
print(drop_X_valid.columns)
print(drop_X_test.columns)

In [ ]:
model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=max_depth,
    random_state=random_state
)

pipeline = Pipeline(steps=[
    ('model', model)
])

def score_dataset(X_train, X_valid, y_train, y_valid):
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)

    print("Accuracy:", accuracy_score(y_valid, predictions))
    print(classification_report(y_valid, predictions))

    print("Accuracy:", accuracy_score(y_valid, predictions))
    print("Precision:", precision_score(y_valid, predictions))
    print("Recall:", recall_score(y_valid, predictions))
    print("F1:", f1_score(y_valid, predictions))
    print("Confusion Matrix:\n", confusion_matrix(y_valid, predictions))


In [ ]:
score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)

In [ ]:
X_test.head()

In [ ]:
ids = drop_X_test.PassengerId
preds_test = pipeline.predict(drop_X_test.drop('PassengerId', axis=1))
output = pd.DataFrame({'PassengerId': ids,
                       'Survived': preds_test})
output.to_csv('submission.csv', index=False)

In [ ]:
plt.figure(figsize=(14, 8))

plot_tree(
    model,
    feature_names=X.columns,
    class_names=["No", "Yes"],
    filled=True,
    rounded=True
)

plt.show()